In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import h5py
import os
import sys
sys.path.append('C:/Users/wg984/Wolfgang/repos/sleep_research_io')
sys.path.append('/home/wolfgang/repos/sleep_research_io')
from sleep_research_functions import index_to_datetime_sleepdata, load_sleep_data, write_to_hdf5_file, read_in_routine, load_bm_data_aligned, merge_bm_and_airgo, airgo_resample_preprocess, bm_resample_interpolate
sys.path.append('C:/Users/wg984/Wolfgang/repos/Bedmaster-ICU-tools/code')
sys.path.append('/home/wolfgang/repos/Bedmaster-ICU-tools/code')
from research_bm_tools import BMR_load, get_metadata
from resample_BMR import remove_non_monotonic_data
from edw_functions import get_vitals, save_edw_vitals_df_for_each_mrn, get_edw_oxygen, get_vitals_single_mrn, get_edw_oxygen_single_mrn
from datetime import datetime, timedelta

In [17]:
### covid-19 respiration data:

study_table = '/media/ssd2/Covid19_Respiration/Database Skeleton.xlsx'
study_table = pd.read_excel(study_table)
study_table.StudyID = study_table.StudyID.apply(lambda x: str(x).zfill(3))

mapping_df = pd.DataFrame()
mapping_df['study_id'] = study_table['StudyID']
mapping_df['mrn'] = study_table['MRN']
mapping_df['bm_file'] = mapping_df['mrn'].apply(lambda x: 'MRN_'+ str(x) + '.h5')
mapping_df['airgo_file'] = mapping_df['study_id'].apply(lambda x: 'airgo_'+ str(x) + '.csv')

bm_files_dir = '/media/mad3/Projects/covid_data/h5_files'
airgo_files_dir = '/media/ssd2/Covid19_Respiration/Data_Analysis/Airgo_Features'
output_dir = '/media/ssd2/Covid19_Respiration/Data_Analysis/Combined_Data'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    
mapping_df['bm_file_path'] = mapping_df['bm_file'].apply(lambda x: os.path.join(bm_files_dir, x))
mapping_df['airgo_file_path'] = mapping_df['airgo_file'].apply(lambda x: os.path.join(airgo_files_dir, x))

# EDW data: for EDW, we use the raw output from EDW queries, MRN selection happs within loading function.
data_dir = '/media/mad3/Projects/covid_data/EDW'
cohort_name = 'airgo_covid_2020_06_30'
vitals_file_path = os.path.join(data_dir, f'edw_{cohort_name}_vitals.csv')
oxygen_supplement_file_path = os.path.join(data_dir, f'edw_{cohort_name}_oxygen_supplement.csv')
mapping_df['edw_vitals_file_path'] = vitals_file_path
mapping_df['edw_oxygen_supplement_file_path'] = oxygen_supplement_file_path

mapping_df['output_file_path'] = mapping_df['study_id'].apply(lambda x: os.path.join(output_dir, f'cov_resp_{x}.h5'))

if 1:
    mapping_df.to_csv('data_mapping_table_covid.csv', index=False)

'/media/ssd2/Covid19_Respiration/Data_Analysis/Combined_Data/cov_resp_001.h5'

In [ ]:
# ecg_dir = '/media/mad3/Projects/ICU_SLEEP_STUDY/data/data_analysis/BMR_resampled_ECG/'
# mapping_df['ecg_file_path'] = mapping_df['study_id'].apply(lambda x: ecg_dir + f'ECG_{x}.h5')